In [ ]:
# 1. Setup
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Set style for better plots
sns.set(style="whitegrid", palette="colorblind", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)

# 2. Supervised Fine-Tuning (SFT)
# 2.1 Data Preparation
currently only placeholder data

In [ ]:
import glob
import os
import re

# Find all out-shakespeare-*** directories
logs_dir = 'logs'
out_dirs = sorted(glob.glob(os.path.join(logs_dir, 'PLACEHOLDER-*')))

# Parse the directory names and train.out files
data = []
for out_dir in out_dirs:
    dir_name = os.path.basename(out_dir)
    # Extract parameters from directory name: PLACEHOLDER-{n_layer}-{n_embd}-{dataset_size}
    match = re.match(r'PLACEHOLDER-(\d+)-(\d+)-([\d.]+)', dir_name)
    if not match:
        continue
    
    n_layer, n_embd, dataset_size = match.groups()
    train_file = os.path.join(out_dir, 'train.out')
    
    if not os.path.exists(train_file):
        continue
    
    # Parse train.out to extract training info
    val_losses = []
    train_losses = []
    num_parameters = None
    tokens_per_iteration = None
    max_iters = None
    
    with open(train_file, 'r') as f:
        for line in f:
            # Parse number of parameters
            if 'number of parameters:' in line:
                match = re.search(r'number of parameters: (\d+)', line)
                if match:
                    num_parameters = int(match.group(1))
            
            # Parse tokens per iteration
            if 'tokens per iteration will be:' in line:
                match = re.search(r'tokens per iteration will be: ([\d,]+)', line)
                if match:
                    tokens_per_iteration = int(match.group(1).replace(',', ''))
            
            # Parse max_iters
            if 'max_iters = ' in line:
                match = re.search(r'max_iters = ([\d.]+)', line)
                if match:
                    max_iters = float(match.group(1))
            
            # Parse validation and training losses
            match = re.search(r'step \d+: train loss ([\d.]+), val loss ([\d.]+)', line)
            if match:
                train_loss = float(match.group(1))
                val_loss = float(match.group(2))
                train_losses.append(train_loss)
                val_losses.append(val_loss)
    
    if val_losses and num_parameters and tokens_per_iteration and max_iters:
        # Calculate FLOPs: 6 * num_parameters * total_tokens
        # (6 accounts for forward, backward, and parameter updates)
        total_tokens = tokens_per_iteration * int(max_iters)
        flops = 6 * num_parameters * total_tokens
        
        # Use final validation loss
        final_val_loss = val_losses[-1]
        final_train_loss = train_losses[-1]
        
        data.append({
            'dir_name': dir_name,
            'model_size': f'{n_layer}L-{n_embd}E',
            'dataset_size': float(dataset_size),
            'val_loss': final_val_loss,
            'train_loss': final_train_loss,
            'n_layer': int(n_layer),
            'n_embd': int(n_embd),
            'num_parameters': num_parameters,
            'tokens_per_iteration': tokens_per_iteration,
            'max_iters': int(max_iters),
            'total_tokens': total_tokens,
            'flops': flops,
        })

df_scaling = pd.DataFrame(data)
print(f"Loaded {len(df_scaling)} configurations from {logs_dir}")
print(df_scaling[['model_size', 'dataset_size', 'val_loss', 'train_loss', 'num_parameters', 'flops']].to_string())

# 2.2 Results Table


In [ ]:
print("\nSFT Results:")
display(df_sft)

# 2.3 Training Curves

In [ ]:
train_loss = [3.2, 2.9, 2.7, 2.5, 2.4]
val_loss = [3.2, 2.95, 2.8, 2.7, 2.6]
steps = list(range(len(train_loss)))

plt.figure(figsize=(12, 6))
plt.plot(steps, train_loss, label="Training Loss", marker="o", linewidth=2.5)
plt.plot(steps, val_loss, label="Validation Loss", marker="o", linewidth=2.5)
plt.title("Fine-Tuning Loss vs. Steps")
plt.ylabel("Loss")
plt.xlabel("Steps")
plt.legend()
plt.grid(True)
plt.show()